# Quering the SOSA graph

In [1]:
# imports

import json
import time
import pandas as pd

from rdflib import Graph
import uuid
from rdflib import Namespace
from rdflib import FOAF, DCTERMS, SKOS, VANN, TIME, RDF, RDFS, XSD, OWL, Literal

In [2]:
g = Graph()
g.parse('/home/brieuc/Documents/dev/abromics/abromics-kg/dev/analysis/sosa-graph.ttl', format='ttl')

<Graph identifier=N63379b49447a40cb8c719570197c3b8f (<class 'rdflib.graph.Graph'>)>

## Quering the sosa graph

Check the journal to see all the queries to test on the SOSA graph

- Get all the resistance genes for a sample
- Rank the resistance genes found for a sample by the coverage percentage
- Get all the antibiotic, a specific strain is resistance to
- Get all the samples where a specific strain is present (filtering by mlst and by specie name)
- Get uniprot id of an antitbiotic and as well as it's molecular formula and a link to it's 3d structure
- Get the most spread resistance genes present in a certain geospacial area

In [3]:
## Q1 of hackmd
## Get all the resistance gen for a sample

query = """
    PREFIX sosa: <http://www.w3.org/ns/sosa/>
    PREFIX schema: <https://schema.org/>
    PREFIX abromics: <https://abromics.fr/>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    PREFIX obo: <http://purl.obolibrary.org/obo/>
    PREFIX aro: <http://purl.obolibrary.org/obo/aro.owl#>
    PREFIX go: <http://purl.org/obo/owl/GO#>
    PREFIX uniprot: <http://purl.uniprot.org/uniprot/>
    PREFIX geo: <http://www.w3.org/2003/01/geo/wgs84_pos#>

    SELECT DISTINCT ?resGeneNames
    WHERE {
        ?sample a sosa:FeatureOfInterest .
        ?sample rdf:type sio:001050 ;
            schema:identifier "0001" .
        
        ?resGenesProperty a sosa:ObservableProperty .
        ?resGenesProperty rdfs:label "Antibiotic resistant gene names"^^xsd:string .
        ?observations sosa:hasObservableProperty ?resGenesProperty .
        ?observations sosa:hasFeatureOfInterest ?sample .
        ?observations sosa:hasResult ?results .
        ?results sosa:hasSimpleResult ?resGeneNames .
    }
"""

res = g.query(query)

for row in res:
    print(f"{row.resGeneNames}")

blaTEM-1A
tet(A)
sul1
qacE
dfrA1
aadA1


In [4]:
## Get all the resistance genes for a sample (with id = 0001) and rank them with their according to their coverage

query = """
    PREFIX sosa: <http://www.w3.org/ns/sosa/>
    PREFIX schema: <https://schema.org/>
    PREFIX abromics: <https://abromics.fr/>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    PREFIX obo: <http://purl.obolibrary.org/obo/>
    PREFIX aro: <http://purl.obolibrary.org/obo/aro.owl#>
    PREFIX go: <http://purl.org/obo/owl/GO#>
    PREFIX uniprot: <http://purl.uniprot.org/uniprot/>
    PREFIX geo: <http://www.w3.org/2003/01/geo/wgs84_pos#>
    
    SELECT DISTINCT ?geneNames ?geneCoverage 
    WHERE {
        ?sample a sosa:FeatureOfInterest .
        ?sample schema:identifier "0001" .
        ?resGenesProperty a sosa:ObservableProperty .
        ?resGenesProperty rdfs:label "Antibiotic resistant gene coverage"^^xsd:string .
        ?observations sosa:hasObservableProperty ?resGenesProperty .
        ?observations sosa:hasFeatureOfInterest ?sample .
        ?genes a sosa:FeatureOfInterest .
        ?genes a go:Gene .
        ?observations sosa:hasFeatureOfInterest ?genes .
        ?genes rdfs:label ?geneNames .
        ?observations sosa:hasResult ?results .
        ?results sosa:hasSimpleResult ?geneCoverage .
    }
    ORDER BY DESC(?geneCoverage)
"""

res = g.query(query)

for row in res:
    print(f"{row.geneNames} - {row.geneCoverage}")

blaTEM-1A - 100
dfrA1 - 100
sul1 - 97.95
tet(A) - 97.8
qacE - 84.68
aadA1 - 76.17


In [5]:
# Q2 of hackmd (all samples)

query = """
    PREFIX sosa: <http://www.w3.org/ns/sosa/>
    PREFIX schema: <https://schema.org/>
    PREFIX abromics: <https://abromics.fr/>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    PREFIX obo: <http://purl.obolibrary.org/obo/>
    PREFIX aro: <http://purl.obolibrary.org/obo/aro.owl#>
    PREFIX go: <http://purl.org/obo/owl/GO#>
    PREFIX uniprot: <http://purl.uniprot.org/uniprot/>
    PREFIX geo: <http://www.w3.org/2003/01/geo/wgs84_pos#>
    PREFIX sio: <http://semanticscience.org/resource/>

    SELECT DISTINCT ?gene_name ?coverage ?s_id WHERE {
    
        ?sample schema:identifier ?s_id ;
            a sio:001050 .
        
        ?coverage_prop a sosa:ObservableProperty ;
                rdfs:label "Antibiotic resistant gene coverage"^^xsd:string .
                
        ?observations sosa:hasObservableProperty ?coverage_prop ;
            sosa:hasFeatureOfInterest ?sample ;
            sosa:hasResult/sosa:hasSimpleResult ?coverage .
        
        ?genes a sosa:FeatureOfInterest ;
            a go:Gene .
            
        ?observations sosa:hasFeatureOfInterest ?genes .
        ?genes rdfs:label ?gene_name .
    } 
    ORDER BY DESC(?coverage) 
    LIMIT 10 # Top-10 most covered
"""

res = g.query(query)

for row in res:
    print(f"{row.gene_name} - {row.coverage} - {row.s_id}")

blaTEM-1A - 100 - 0001
tet(A) - 100 - 0002
sul1 - 100 - 0002
dfrA1 - 100 - 0001
dfrA1 - 100 - 0002
mcr-1.1 - 100 - 0002
dfrA36 - 100 - 0002
sul2 - 100 - 0002
aph(3'')-Ib - 100 - 0002
blaOXA-1 - 100 - 0002


In [6]:
## Q3: List the TOP-K antibiotic resistance genes by coverage in a geographical region of interest

query = """
    PREFIX sosa: <http://www.w3.org/ns/sosa/>
    PREFIX schema: <https://schema.org/>
    PREFIX abromics: <https://abromics.fr/>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    PREFIX obo: <http://purl.obolibrary.org/obo/>
    PREFIX aro: <http://purl.obolibrary.org/obo/aro.owl#>
    PREFIX go: <http://purl.org/obo/owl/GO#>
    PREFIX uniprot: <http://purl.uniprot.org/uniprot/>
    PREFIX geo: <http://www.w3.org/2003/01/geo/wgs84_pos#>

    SELECT DISTINCT ?gene_name ?coverage ?s_id ?lat ?long ?alt 
    WHERE {
    
        ?location a sosa:FeatureOfInterest ;
            geo:lat "47.209110"^^xsd:decimal ;
            geo:long "-1.553479"^^xsd:decimal ;
            geo:alt "10.00"^^xsd:decimal .
            
        ?location geo:lat ?lat ;
            geo:long ?long ;
            geo:alt ?alt .
    
        ?sample schema:identifier ?s_id ;
            a sio:001050 .
        
        ?coverage_prop a sosa:ObservableProperty ;
                rdfs:label "Antibiotic resistant gene coverage"^^xsd:string .
                
        ?observations sosa:hasObservableProperty ?coverage_prop ;
            sosa:hasFeatureOfInterest ?sample ;
            sosa:hasFeatureOfInterest ?location ;
            sosa:hasResult/sosa:hasSimpleResult ?coverage .
        
        ?genes a sosa:FeatureOfInterest ;
            a go:Gene .
            
        ?observations sosa:hasFeatureOfInterest ?genes .
        ?genes rdfs:label ?gene_name .
    }
    ORDER BY DESC(?coverage) 
    LIMIT 20 
"""

res = g.query(query)

for row in res:
    print(f"{row.gene_name} - {row.coverage} - {row.s_id} - {row.lat} - {row.long} - {row.alt}")

tet(A) - 100 - 0002 - 47.209110 - -1.553479 - 10.00
sul1 - 100 - 0002 - 47.209110 - -1.553479 - 10.00
dfrA1 - 100 - 0002 - 47.209110 - -1.553479 - 10.00
mcr-1.1 - 100 - 0002 - 47.209110 - -1.553479 - 10.00
dfrA36 - 100 - 0002 - 47.209110 - -1.553479 - 10.00
sul2 - 100 - 0002 - 47.209110 - -1.553479 - 10.00
aph(3'')-Ib - 100 - 0002 - 47.209110 - -1.553479 - 10.00
blaOXA-1 - 100 - 0002 - 47.209110 - -1.553479 - 10.00
aph(6)-Id - 100 - 0002 - 47.209110 - -1.553479 - 10.00
floR - 99.92 - 0002 - 47.209110 - -1.553479 - 10.00
aph(3')-Ia - 99.75 - 0002 - 47.209110 - -1.553479 - 10.00
aadA1 - 98.61 - 0002 - 47.209110 - -1.553479 - 10.00
qacE - 84.68 - 0002 - 47.209110 - -1.553479 - 10.00


In [7]:
## Q4: Common antibiotic resistance genes between two time points, for all available samples

query = """
    PREFIX sosa: <http://www.w3.org/ns/sosa/>
    PREFIX schema: <https://schema.org/>
    PREFIX abromics: <https://abromics.fr/>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    PREFIX obo: <http://purl.obolibrary.org/obo/>
    PREFIX aro: <http://purl.obolibrary.org/obo/aro.owl#>
    PREFIX go: <http://purl.org/obo/owl/GO#>
    PREFIX uniprot: <http://purl.uniprot.org/uniprot/>
    PREFIX geo: <http://www.w3.org/2003/01/geo/wgs84_pos#>
    
    SELECT ?resGeneName
    WHERE {

        ?resGenesProperty a sosa:ObservableProperty .
        ?resGenesProperty rdfs:label "Antibiotic resistant gene names"^^xsd:string .
        ?observations sosa:hasObservableProperty ?resGenesProperty .
        
        ?observations sosa:hasResult/sosa:resultTime ?resTime .
        FILTER(?resTime > "2024-06-05T12:00:00Z"^^xsd:dateTime && ?resTime < "2024-07-01T12:00:00Z"^^xsd:dateTime).
        
        ?observations sosa:hasResult ?results .
        ?results sosa:hasSimpleResult ?resGeneName .
    }
"""

res = g.query(query)

for row in res:
    print(f"{row.resGeneName}")

mcr-1.1
tet(A)
dfrA36
floR
sul2
aph(3')-Ia
sul1
qacE
aadA1
aph(6)-Id
aph(3'')-Ib
blaOXA-1
dfrA1


In [8]:
## Q6: Get all the antibiotics a specific specie (defined by its ncbi taxonomic id) is resistant to

query = """
    PREFIX sosa: <http://www.w3.org/ns/sosa/>
    PREFIX schema: <https://schema.org/>
    PREFIX abromics: <https://abromics.fr/>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    PREFIX obo: <http://purl.obolibrary.org/obo/>
    PREFIX aro: <http://purl.obolibrary.org/obo/aro.owl#>
    PREFIX go: <http://purl.org/obo/owl/GO#>
    PREFIX uniprot: <http://purl.uniprot.org/uniprot/>
    PREFIX geo: <http://www.w3.org/2003/01/geo/wgs84_pos#>

    SELECT DISTINCT ?antibiotics 
    WHERE {
        ?strain a aro:Strain ;
            a sosa:FeatureOfInterest .
        ?strain obo:RO_0002162 <http://purl.uniprot.org/taxonomy/562> . # Get the E.Coli taxon from ncbi
        ?strain aro:confers_resistance_to_antibiotic ?antibiotics
    }
"""

res = g.query(query)

for row in res:
    print(f"{row.antibiotics}")

http://purl.obolibrary.org/obo/aro.owl#ampicillin
http://purl.obolibrary.org/obo/aro.owl#tetracycline
http://purl.obolibrary.org/obo/aro.owl#sulfisoxazole
http://purl.obolibrary.org/obo/aro.owl#trimethoprim
http://purl.obolibrary.org/obo/aro.owl#streptomycin
http://purl.obolibrary.org/obo/aro.owl#chloramphenicol
http://purl.obolibrary.org/obo/aro.owl#tcolistin
http://purl.obolibrary.org/obo/aro.owl#kanamycin


In [9]:
## Q7: Find all the strain with the same antibiotic resistance gene found

## change "sul1" to "sul2" to showcase that the filtering process is working as the strain 1
## contains "sul1" but not "sul2" 

query = """
    PREFIX sosa: <http://www.w3.org/ns/sosa/>
    PREFIX schema: <https://schema.org/>
    PREFIX abromics: <https://abromics.fr/>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    PREFIX obo: <http://purl.obolibrary.org/obo/>
    PREFIX aro: <http://purl.obolibrary.org/obo/aro.owl#>
    PREFIX go: <http://purl.org/obo/owl/GO#>
    PREFIX uniprot: <http://purl.uniprot.org/uniprot/>
    PREFIX geo: <http://www.w3.org/2003/01/geo/wgs84_pos#>
    
    SELECT ?strains ?latitude ?longitude ?altitude
    WHERE {
        ?strains a aro:Strain .
            
        ?gene rdfs:label "sul1" .
        ?geneObservableProperty a sosa:ObservableProperty ;
            rdfs:label "Antibiotic resistant gene names"^^xsd:string .
        ?observations sosa:hasFeatureOfInterest ?gene ;
            sosa:hasObservableProperty ?geneObservableProperty ;
            sosa:hasFeatureOfInterest ?strains ;
            sosa:hasFeatureOfInterest ?location .
        ?location geo:lat ?latitude ;
            geo:long ?longitude ;
            geo:alt ?altitude .
    }
"""

res = g.query(query)

for row in res:
    print(f"{row.strains} - {row.latitude} - {row.longitude} - {row.altitude}")

file:///home/brieuc/Documents/dev/abromics/abromics-kg/dev/analysis/strain_1 - 35.8648067 - -120.6195831 - 12.75
file:///home/brieuc/Documents/dev/abromics/abromics-kg/dev/analysis/strain_2 - 47.209110 - -1.553479 - 10.00
